# Project Vyapaar | Notebook 01: Data Cleaning 🔎

**Team:** ____________________

**Lead for this part:** Member 1 (Data Analyst)

Can we trust the data? Here we find the data problems, fix them and write down every decision.

### How to run this notebook
**On Google Colab:**
1. Upload this notebook to Colab (File → Upload notebook).
2. Click **Runtime → Run all**.
3. When the "Choose Files" button appears, select the 4 data files: `orders.csv`, `products.csv`, `reviews.csv`, `marketing.csv`.

**On Jupyter (laptop):** keep the 4 csv files in the same folder as this notebook (or in a `data` folder next to it) and run all cells.

## Step 0: Load the data

In [ ]:
import pandas as pd
import plotly.express as px
import os

# show numbers with 2 decimals and commas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

In [ ]:
# This small function finds a csv file whether it is in this folder or in a "data" folder
def find_file(name):
    for folder in ["", "data/", "../data/"]:
        if os.path.exists(folder + name):
            return folder + name
    return None

# If the files are not there we are probably on Google Colab, so ask to upload them
if find_file("orders.csv") is None:
    from google.colab import files
    print("Please upload orders.csv, products.csv, reviews.csv and marketing.csv")
    files.upload()

In [ ]:
# encoding="utf-8-sig" removes a hidden character at the start of orders.csv and reviews.csv
orders = pd.read_csv(find_file("orders.csv"), encoding="utf-8-sig")
products = pd.read_csv(find_file("products.csv"))
marketing = pd.read_csv(find_file("marketing.csv"))

# keep_default_na=False stops pandas from turning the word "None" into a blank.
# In reviews, "None" means "no complaint", so we want to keep it.
reviews = pd.read_csv(find_file("reviews.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""])

print("orders:", orders.shape)
print("products:", products.shape)
print("reviews:", reviews.shape)
print("marketing:", marketing.shape)

---
# Stage 1: Data Detective 🔎
Before making any recommendation, we first check if we can trust the data.

### 1.1 First look

In [ ]:
orders.head()

In [ ]:
products.head()

In [ ]:
reviews.head()

In [ ]:
marketing.head()

In [ ]:
orders.info()

`order_date` is stored as text (object), not as a date. We will convert it later.

### 1.2 Missing values

In [ ]:
print("ORDERS")
print(orders.isnull().sum())
print()
print("REVIEWS")
print(reviews.isnull().sum())

In [ ]:
# which orders have a missing customer or discount?
orders[orders["customer_id"].isnull() | orders["discount_pct"].isnull()]

In [ ]:
reviews[reviews["rating"].isnull()]

**What we decided:**
- **campaign_id** is blank for 1,472 orders. That is too many to be a mistake. It just means the order did not come from a campaign, so we label it **"No campaign"**.
- **O00300** has no customer. The sale is still real, so we keep it and call the customer **"Unknown"**.
- **O00778** has no discount. We can work it out from the revenue (see 1.6).
- **R01201** has no rating. We keep the row (it still shows a late delivery and an escalation), and we do **not** make up a rating.

### 1.3 Duplicates

In [ ]:
print("Duplicate rows in orders:", orders.duplicated().sum())
print("Duplicate rows in products:", products.duplicated().sum())
print("Duplicate rows in reviews:", reviews.duplicated().sum())
print("Duplicate rows in marketing:", marketing.duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)]

In [ ]:
products[products.duplicated(keep=False)]

The duplicates are exact copies (same ID, same values), so they are double entries and we remove them.

⚠️ The product duplicate (**P040**) is important. If we don't remove it, every P040 order will be counted **twice** when we join the tables.

### 1.4 Spelling and category problems

In [ ]:
products["category"].value_counts()

In [ ]:
orders["sales_channel"].value_counts()

In [ ]:
products[(products["category"] == "Electronic") | (products["subcategory"] == "skin care")]

- **P011** is called "Hair Care Item 011", but its category says "Electronic".
- **P074** is called "Small Appliances Item 074", but its subcategory says "skin care".
- "web site" is just another spelling of "Website".

To be sure about the two products, we checked the **mark-up** (list price ÷ cost). Each category seems to have its own fixed mark-up:

In [ ]:
products["markup"] = products["list_price_inr"] / products["unit_cost_inr"]
products.groupby("category")["markup"].mean()

In [ ]:
products[products["product_id"].isin(["P011", "P074"])][["product_id", "product_name", "category", "subcategory", "markup"]]

P011's mark-up (1.92) matches **Beauty**, and P074's (1.47) matches **Home & Kitchen**. So the names were right and the labels were wrong:
- P011 → category = **Beauty**
- P074 → subcategory = **Small Appliances**

### 1.5 Do the money columns add up?
We re-calculated revenue ourselves: **revenue = quantity × price × (1 − discount)**

In [ ]:
check = orders.copy()
check["my_revenue"] = check["quantity"] * check["unit_price_inr"] * (1 - check["discount_pct"])
check["difference"] = (check["my_revenue"] - check["revenue_inr"]).abs()

print("Orders where revenue does not match:", (check["difference"] > 1).sum())
print("Orders where profit is not revenue - cost:", ((check["revenue_inr"] - check["cost_inr"] - check["profit_inr"]).abs() > 1).sum())

Everything matches 👍 so we can trust the revenue, cost and profit numbers.

### 1.6 Working out the missing discount
Since revenue = qty × price × (1 − discount), the missing discount = **1 − revenue ÷ (qty × price)**

In [ ]:
row = orders[orders["discount_pct"].isnull()]
print(1 - row["revenue_inr"] / (row["quantity"] * row["unit_price_inr"]))
print()
print("Discount levels used by the company:", sorted(orders["discount_pct"].dropna().unique().tolist()))

The answer is **0.30**, which is one of the company's normal discount levels (0, 5, 10, 15, 20, 30, 40%). So this is the real value, not a guess.

### 1.7 Strange values

In [ ]:
print("Dates from", orders["order_date"].min(), "to", orders["order_date"].max())
print("Quantity from", orders["quantity"].min(), "to", orders["quantity"].max())
print("Ratings:", sorted(reviews["rating"].dropna().unique().tolist()))
print("Orders with NEGATIVE profit:", (orders["profit_inr"] < 0).sum())

**313 orders have negative profit.** We first thought this was an error, but 1.5 showed the maths is correct. These are real sales where the discount was bigger than the margin, so it is a **business finding, not an error**. We keep them.

### 1.8 Do the tables connect?

In [ ]:
print("Orders with a product that doesn't exist:", (~orders["product_id"].isin(products["product_id"])).sum())
print("Orders without a review:", (~orders["order_id"].isin(reviews["order_id"])).sum())

# we don't have customers.csv, so can we use region_at_order as the customer's region?
regions_per_customer = orders.groupby("customer_id")["region_at_order"].nunique()
print("Customers who ordered from more than 1 region:", (regions_per_customer > 1).sum())

Every customer always orders from the **same region**, so we can use `region_at_order` as the customer's region (we did not get `customers.csv`).

### 1.9 Apply the cleaning

In [ ]:
# 1. remove exact duplicate rows
orders = orders.drop_duplicates()
products = products.drop_duplicates()
reviews = reviews.drop_duplicates()

# 2. fix the 2 wrongly labelled products
products.loc[products["product_id"] == "P011", "category"] = "Beauty"
products.loc[products["product_id"] == "P074", "subcategory"] = "Small Appliances"
products = products.drop(columns="markup")

# 3. fix spelling of sales channel
orders["sales_channel"] = orders["sales_channel"].replace("web site", "Website")

# 4. fill the 1 missing discount using: discount = 1 - revenue / (quantity x price)
orders["discount_pct"] = orders["discount_pct"].fillna(
    (1 - orders["revenue_inr"] / (orders["quantity"] * orders["unit_price_inr"])).round(2))

# 5. label blanks instead of deleting them
orders["customer_id"] = orders["customer_id"].fillna("Unknown")
orders["campaign_id"] = orders["campaign_id"].fillna("No campaign")

# 6. text -> dates
orders["order_date"] = pd.to_datetime(orders["order_date"])
marketing["start_date"] = pd.to_datetime(marketing["start_date"])
marketing["end_date"] = pd.to_datetime(marketing["end_date"])

print("orders:", orders.shape, "| products:", products.shape, "| reviews:", reviews.shape)

### 1.10 Our cleaning log (Problem → Action → Reason)

| # | Problem found | Action taken | Reason |
|---|---|---|---|
| 1 | Hidden character at the start of orders.csv and reviews.csv | Read with `encoding="utf-8-sig"` | Otherwise the `order_id` column name is wrong and joins fail |
| 2 | pandas turns "None" (no complaint) into a blank | Read reviews with `keep_default_na=False` | "None" is real information: 2,640 orders had no complaint |
| 3 | Order O00501 appears twice | Removed 1 copy | Double entry, would add ₹6,706 of fake revenue |
| 4 | Review R00334 appears twice | Removed 1 copy | One review per order |
| 5 | Product P040 appears twice | Removed 1 copy before joining | Would double count all P040 orders |
| 6 | P011 "Hair Care Item" labelled "Electronic" | Changed to Beauty | Name + mark-up (1.92) match Beauty |
| 7 | P074 "Small Appliances Item" labelled "skin care" | Changed to Small Appliances | Name + mark-up (1.47) match Home & Kitchen |
| 8 | Sales channel "web site" | Changed to "Website" | Same channel, different spelling |
| 9 | O00778 missing discount | Calculated it = 0.30 | Worked out from revenue, so not a guess |
| 10 | O00300 missing customer | Kept, customer = "Unknown" | The sale is real |
| 11 | R01201 missing rating | Kept, rating left blank | We don't invent ratings |
| 12 | 1,472 orders with no campaign | Labelled "No campaign" | Blank means no campaign, not an error |
| 13 | 313 orders with negative profit | Kept | Maths is correct, caused by big discounts |
| 14 | Dates stored as text | Converted to dates | Needed for monthly / campaign analysis |
| 15 | customers.csv and delivery.csv not given to us | Used region_at_order and "Late delivery" complaints instead | Each customer always orders from one region |

### 1.11 One clean table

In [ ]:
# join orders + products + reviews using the ID columns
df = orders.merge(products, on="product_id").merge(reviews, on="order_id")
df["margin_%"] = df["profit_inr"] / df["revenue_inr"] * 100

print("Clean table:", df.shape)
df.to_csv("vyapaar_clean.csv", index=False)   # on Colab you can download it from the folder icon
df.head()

✅ **Stage 1 result:** after a few small fixes, the sales and money data can be trusted.